# Answer Preparation — Top-100 Retrieval per Query

**Where we are:** Between Step 4 (dense retrieval) and Step 5 (KG-enhanced reranking)  
**What we do:** For each query, retrieve the top-100 most similar passages across all FAISS shards  
**How it connects:** These top-100 passages per query will be the input for KG-based reranking (Step 5)

### Pipeline

1. Encode all 31,372 queries with Contriever → save `query_embeddings.npy`
2. For each shard (0–8): load FAISS index on GPU → batch search all queries → save per-shard top-100
3. Merge per-shard results → keep global top-100 per query → save `top100_merged.parquet`

### Output structure

```
data/NQ_answer/
├── query_embeddings.npy          ← (31372, 768) float32, encoded once
├── shard_00/
│   └── top100_shard.parquet      ← query_id, passage_id, score, rank
├── shard_01/
│   └── top100_shard.parquet
...
└── top100_merged.parquet         ← global top-100 after cross-shard merge
```

## 1. Setup

In [1]:
import numpy as np
import os
import sys
import json
import math
import time
import torch
import polars as pl
from pathlib import Path
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

# faiss-gpu: add DLL paths and conda site-packages
os.add_dll_directory('C:/Users/Utente/miniconda3/Library/bin')
os.add_dll_directory('C:/Users/Utente/miniconda3/Library/lib')
sys.path.insert(0, 'C:/Users/Utente/miniconda3/Lib/site-packages')

import faiss

# faiss.get_num_gpus() → int: returns the number of CUDA GPUs visible to FAISS
print(f"faiss {faiss.__version__} — GPU devices: {faiss.get_num_gpus()}")
print(f"torch {torch.__version__} — CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    # torch.cuda.get_device_name(device_id) → str: human-readable GPU name (e.g. "NVIDIA RTX 4090")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

faiss 1.12.0 — GPU devices: 1
torch 2.10.0+cu128 — CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Ti


## 2. Configuration

In [2]:
# --- Paths ---
QUERIES_PATH = Path("data/NQ_question/qa_all_entities.jsonl")
FAISS_DIR    = Path("data/faiss_index")      # where shard_XX.faiss / shard_XX_ids.npy live
OUTPUT_DIR   = Path("data/NQ_answer")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Model (same as embedding.ipynb) ---
MODEL_NAME = "facebook/contriever"
MAX_LENGTH = 512

# --- Retrieval ---
TOP_K      = 100   # passages to retrieve per query per shard
N_SHARDS   = 9     # shard_00 .. shard_08
BATCH_SIZE = 256   # queries per encoding batch (queries are shorter than passages)

# --- Device ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## 3. Load Queries

Load all 31,372 queries from `qa_all_entities.jsonl`.  
Each query has entities in both question and answers — these will be used later for KG reranking.  
We use the line index (0-based) as `query_id`.

In [3]:
# load all queries from the JSONL file
queries = []
with open(QUERIES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        queries.append(json.loads(line))

query_texts = [q["question"] for q in queries]
n_queries = len(query_texts)

print(f"Loaded {n_queries:,} queries")
print(f"Example: '{query_texts[0]}'")

Loaded 31,372 queries
Example: 'where do you cross the arctic circle in norway'


## 4. Encode Queries with Contriever

Same model and mean-pooling as passage encoding (`embedding.ipynb`).  
Queries are encoded **once** and reused for all shard searches.

In [4]:
# AutoTokenizer.from_pretrained(model_name) → PreTrainedTokenizer
#   Downloads (or loads from cache) the tokenizer config for the given HF model.
#   Returns a tokenizer that can convert text → token IDs matching the model's vocabulary.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# AutoModel.from_pretrained(model_name) → PreTrainedModel
#   Downloads (or loads from cache) the model weights and architecture.
#   For Contriever: returns a BERT-based encoder that outputs contextual embeddings.
# .to(device) moves all parameters to the specified device (CPU or CUDA GPU).
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

# model.eval() switches the model to inference mode:
#   - Disables dropout layers: during training, dropout randomly zeroes a fraction
#     of neuron activations at each forward pass (e.g. 10%) to prevent overfitting.
#     In eval mode all activations are kept, so the output is deterministic.
#   - Disables batch-norm training updates: batch-norm layers stop tracking
#     running statistics and use the stored mean/variance instead.
# Without eval(), the same input could produce different embeddings on each call.
model.eval()
print(f"Model loaded on {DEVICE}")

Model loaded on cuda


In [6]:
def mean_pooling(model_output, attention_mask):
    """Contriever-style mean pooling: average non-padding tokens."""
    # model_output.last_hidden_state → (batch, seq_len, 768)
    #   The contextual embedding for every token in the input sequence.
    last_hidden = model_output.last_hidden_state

    # masked_fill(mask, value): where mask is True, replace with value.
    #   ~attention_mask.unsqueeze(-1).bool() → True for padding positions.
    #   Zeroes out padding token embeddings so they don't affect the average.
    last_hidden = last_hidden.masked_fill(
        ~attention_mask.unsqueeze(-1).bool(), 0.0
    )

    # Sum non-padding embeddings along seq_len (dim=1) → (batch, 768),
    # then divide by the count of real tokens per sample → mean embedding.
    # attention_mask.sum(dim=1, keepdim=True) → (batch, 1): number of non-padding tokens.
    emb = last_hidden.sum(dim=1) / attention_mask.sum(dim=1, keepdim=True)
    return emb


# @torch.no_grad(): disables gradient computation for all operations inside.
#   Reduces memory usage (no computation graph stored) and speeds up inference.
@torch.no_grad()
def encode_batch(texts, model, tokenizer, device, max_length=MAX_LENGTH):
    """Encode a list of strings into float32 numpy embeddings."""
    # tokenizer(texts, ...) → BatchEncoding (dict-like with input_ids, attention_mask)
    #   max_length: truncate sequences longer than this (in tokens)
    #   padding=True: pad shorter sequences to the longest in the batch
    #   truncation=True: cut sequences exceeding max_length
    #   return_tensors="pt": return PyTorch tensors (vs "np" for numpy, "tf" for TF)
    # .to(device): move all tensors to GPU/CPU
    inputs = tokenizer(
        texts,
        max_length=max_length,
        padding=True,
        truncation=True,
        return_tensors="pt",
    ).to(device)

    # model(**inputs) → BaseModelOutput
    #   Unpacks input_ids and attention_mask as kwargs to the forward pass.
    #   Returns .last_hidden_state (batch, seq_len, 768) among other attributes.
    output = model(**inputs)
    emb = mean_pooling(output, inputs["attention_mask"])

    # .cpu().numpy() → moves tensor to CPU RAM, then converts to numpy float32 array
    return emb.cpu().numpy()

In [7]:
query_emb_path = OUTPUT_DIR / "query_embeddings.npy"

if query_emb_path.exists():
    # resume support: reuse previously encoded queries
    print("Loading cached query embeddings ...")
    # np.load(path) → ndarray: reads a .npy binary file back into a numpy array,
    #   preserving the original dtype and shape metadata stored in the file header.
    query_embeddings = np.load(query_emb_path)
    print(f"Loaded {query_embeddings.shape}")
else:
    print(f"Encoding {n_queries:,} queries in batches of {BATCH_SIZE} ...")
    t0 = time.time()
    emb_list = []

    for start in tqdm(range(0, n_queries, BATCH_SIZE), desc="Encoding queries"):
        batch = query_texts[start : start + BATCH_SIZE]
        emb_list.append(encode_batch(batch, model, tokenizer, DEVICE))

    # np.concatenate(arrays, axis=0) → ndarray
    #   Joins a list of arrays along the first axis (stacking rows).
    #   Here: list of (batch_size, 768) → single (n_queries, 768) array.
    query_embeddings = np.concatenate(emb_list, axis=0)
    del emb_list

    # np.save(path, array): writes array to a binary .npy file
    #   with a small header (dtype, shape, order) for exact reconstruction via np.load.
    np.save(query_emb_path, query_embeddings)
    elapsed = time.time() - t0
    print(f"Saved {query_emb_path} — shape {query_embeddings.shape} in {elapsed:.1f}s")

# sanity check
assert query_embeddings.shape == (n_queries, 768), \
    f"Shape mismatch: {query_embeddings.shape} vs expected ({n_queries}, 768)"

Encoding 31,372 queries in batches of 256 ...


Encoding queries:   0%|          | 0/123 [00:00<?, ?it/s]

Saved data\NQ_answer\query_embeddings.npy — shape (31372, 768) in 8.0s


## 5. Rebuild Missing FAISS Indices

Some shards may have `.npy` embeddings but no `.faiss` index (e.g., shard_00).  
We rebuild any missing indices before the search loop.

In [8]:
for shard_idx in range(N_SHARDS):
    shard_npy  = FAISS_DIR / f"shard_{shard_idx:02d}.npy"
    index_path = FAISS_DIR / f"shard_{shard_idx:02d}.faiss"

    # skip if index already exists or embeddings not available
    if index_path.exists() or not shard_npy.exists():
        continue

    print(f"Rebuilding FAISS index for shard {shard_idx:02d} ...", end=" ")
    embeddings = np.load(shard_npy)

    # faiss.IndexFlatIP(d) → IndexFlatIP
    #   Creates a flat (brute-force) index for inner-product similarity.
    #   d: dimensionality of vectors (768 for Contriever).
    #   "Flat" = no compression/quantization: exact search, O(n) per query.
    #   "IP" = Inner Product: score = dot(query, passage). Higher = more similar.
    index = faiss.IndexFlatIP(embeddings.shape[1])

    # index.add(vectors: np.ndarray[float32]) → None
    #   Inserts all vectors into the index. vectors shape: (n_vectors, d).
    #   After this call, index.ntotal == n_vectors.
    index.add(embeddings)

    # faiss.write_index(index, path) → None
    #   Serializes the FAISS index (metadata + all vectors) to a binary file on disk.
    faiss.write_index(index, str(index_path))
    print(f"done — {index.ntotal:,} vectors")
    del embeddings, index

print("All available indices ready.")

All available indices ready.


## 6. Per-Shard Retrieval

For each shard:
1. Load the `.faiss` index → move to GPU
2. Batch search: all 31,372 queries × top-100 in one call
3. Map FAISS positions → passage IDs via `shard_XX_ids.npy`
4. Save as `shard_XX/top100_shard.parquet`

Shards missing from disk are silently skipped.

In [9]:
for shard_idx in range(N_SHARDS):
    index_path = FAISS_DIR / f"shard_{shard_idx:02d}.faiss"
    ids_path   = FAISS_DIR / f"shard_{shard_idx:02d}_ids.npy"

    # --- Skip missing shards silently ---
    if not index_path.exists() or not ids_path.exists():
        print(f"Shard {shard_idx:02d}: not on disk, skipping")
        continue

    # --- Resume support: skip already-processed shards ---
    shard_out_dir = OUTPUT_DIR / f"shard_{shard_idx:02d}"
    shard_out_dir.mkdir(parents=True, exist_ok=True)
    parquet_path = shard_out_dir / "top100_shard.parquet"

    if parquet_path.exists():
        print(f"Shard {shard_idx:02d}: already processed, skipping")
        continue

    print(f"\nShard {shard_idx:02d}: loading index ...", end=" ")
    t0 = time.time()

    # faiss.read_index(path) → Index
    #   Deserializes a FAISS index from the binary file written by write_index.
    #   Returns the full index in CPU RAM.
    cpu_index = faiss.read_index(str(index_path))

    # faiss.StandardGpuResources() → GpuResources
    #   Allocates GPU memory workspace (scratch space, temp buffers) for FAISS operations.
    #   One instance per GPU; manages a default 256 MB temp memory pool.
    gpu_res = faiss.StandardGpuResources()

    # faiss.index_cpu_to_gpu(resources, device_id, cpu_index) → GpuIndex
    #   Copies the entire CPU index onto the specified GPU.
    #   device_id=0: first CUDA GPU.
    #   Returns a GPU-backed index with the same search interface but much faster.
    gpu_index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_index)
    del cpu_index

    # load passage ID mapping
    shard_ids = np.load(ids_path)
    print(f"{gpu_index.ntotal:,} vectors on GPU")

    # --- Batch search: all queries at once ---
    # gpu_index.search(queries, k) → (scores, indices)
    #   queries: np.ndarray float32, shape (n_queries, d) — the query vectors
    #   k: int — number of nearest neighbors to return per query
    #   Returns two np.ndarray, both shape (n_queries, k):
    #     scores: float32 — inner product similarities.
    #       Each row is sorted descending: scores[i, 0] is the best match for query i.
    #       For IndexFlatIP, these are raw dot products (not normalized to [0,1]).
    #     indices: int64 — 0-based row positions within this index.
    #       indices[i, j] is the position of the j-th best match for query i.
    #       A value of -1 means fewer than k vectors were found (rare, only if ntotal < k).
    #       These are NOT passage IDs — they must be mapped via shard_ids below.
    print(f"Searching {n_queries:,} queries × top-{TOP_K} ...", end=" ")
    scores, indices = gpu_index.search(query_embeddings, TOP_K)
    print(f"done in {time.time() - t0:.1f}s")

    # --- Map FAISS positions → passage IDs ---
    # indices contains positions within this shard (0, 1, 2, ...)
    # shard_ids maps those positions to the original passage IDs from the corpus
    # numpy advanced indexing: shard_ids[indices] → (n_queries, TOP_K) of real IDs
    passage_ids = shard_ids[indices]  # (n_queries, TOP_K)

    # --- Build result DataFrame ---
    # flatten the 2D arrays into 1D for Polars
    # np.repeat(a, repeats): repeats each element — [0,1,2] repeat 2 → [0,0,1,1,2,2]
    # np.tile(a, reps): tiles the whole array — [0,1,2] tile 2 → [0,1,2,0,1,2]
    query_id_col = np.repeat(np.arange(n_queries, dtype=np.int32), TOP_K)
    rank_col = np.tile(np.arange(TOP_K, dtype=np.int16), n_queries)

    # pl.DataFrame(dict) → DataFrame
    #   Constructs a Polars DataFrame from a dict of {column_name: array}.
    #   Accepts numpy arrays directly; each array becomes one column (zero-copy when possible).
    result_df = pl.DataFrame({
        "query_id":   query_id_col,
        "passage_id": passage_ids.ravel().astype(np.int64),
        "score":      scores.ravel().astype(np.float32),
        "rank":       rank_col,
    })

    # df.write_parquet(path) → None
    #   Writes the DataFrame to Apache Parquet format (columnar, compressed).
    #   Efficient for large tabular data: smaller on disk than CSV, fast to read back.
    result_df.write_parquet(parquet_path)
    print(f"Saved {parquet_path} — {len(result_df):,} rows")

    # free GPU memory before next shard
    del gpu_index, gpu_res, shard_ids, scores, indices, passage_ids, result_df
    # torch.cuda.empty_cache() → None
    #   Releases all unused cached GPU memory held by the PyTorch allocator,
    #   making it available to other GPU operations (here: next FAISS shard load).
    torch.cuda.empty_cache()

print("\nPer-shard retrieval complete.")


Shard 00: loading index ... 5,000,000 vectors on GPU
Searching 31,372 queries × top-100 ... done in 82.0s
Saved data\NQ_answer\shard_00\top100_shard.parquet — 3,137,200 rows
Shard 01: not on disk, skipping
Shard 02: not on disk, skipping
Shard 03: not on disk, skipping

Shard 04: loading index ... 5,000,000 vectors on GPU
Searching 31,372 queries × top-100 ... done in 63.4s
Saved data\NQ_answer\shard_04\top100_shard.parquet — 3,137,200 rows

Shard 05: loading index ... 5,000,000 vectors on GPU
Searching 31,372 queries × top-100 ... done in 52.0s
Saved data\NQ_answer\shard_05\top100_shard.parquet — 3,137,200 rows

Shard 06: loading index ... 5,000,000 vectors on GPU
Searching 31,372 queries × top-100 ... done in 59.9s
Saved data\NQ_answer\shard_06\top100_shard.parquet — 3,137,200 rows

Shard 07: loading index ... 5,000,000 vectors on GPU
Searching 31,372 queries × top-100 ... done in 54.4s
Saved data\NQ_answer\shard_07\top100_shard.parquet — 3,137,200 rows

Shard 08: loading index ... 

## 7. Merge Cross-Shard Results

For each query, we collected up to 9 × 100 = 900 candidates (one set per shard).  
Now we merge them and keep only the **global top-100** by score.

The merge uses Polars `group_by` + `sort` + `head` — efficient on ~28M rows.

In [10]:
merged_path = OUTPUT_DIR / "top100_merged.parquet"

if merged_path.exists():
    print(f"Merged file already exists: {merged_path}")
    # pl.read_parquet(path) → DataFrame
    #   Reads an Apache Parquet file into a Polars DataFrame.
    #   Parquet stores column types in metadata, so no schema inference needed.
    merged_df = pl.read_parquet(merged_path)
else:
    # collect all per-shard parquets
    shard_dfs = []
    for shard_idx in range(N_SHARDS):
        parquet_path = OUTPUT_DIR / f"shard_{shard_idx:02d}" / "top100_shard.parquet"
        if parquet_path.exists():
            df = pl.read_parquet(parquet_path)
            # pl.lit(value) → Expr: creates a literal (constant) expression.
            # .cast(pl.Int8) converts to 8-bit integer (sufficient for shard IDs 0–8).
            # .alias(name) renames the resulting column.
            # df.with_columns(...) adds or replaces columns, returns a new DataFrame.
            df = df.with_columns(pl.lit(shard_idx).cast(pl.Int8).alias("shard_id"))
            shard_dfs.append(df)
            print(f"Loaded shard {shard_idx:02d}: {len(df):,} rows")
        else:
            print(f"Shard {shard_idx:02d}: not available, skipping")

    if not shard_dfs:
        raise RuntimeError("No shard results found — run section 6 first")

    print(f"\nMerging {len(shard_dfs)} shards ...")
    t0 = time.time()

    # pl.concat(dfs) → DataFrame
    #   Vertically stacks a list of DataFrames (like SQL UNION ALL).
    #   All DataFrames must share the same schema (column names and types).
    all_results = pl.concat(shard_dfs)
    del shard_dfs
    print(f"Total candidates: {len(all_results):,}")

    # Pipeline: for each query, keep the top-100 passages by score globally.
    #
    # .sort("score", descending=True) → DataFrame
    #   Sorts all rows by the "score" column in descending order (best first).
    #
    # .group_by("query_id", maintain_order=True) → GroupBy
    #   Groups rows by query_id. maintain_order=True preserves the sort order
    #   within each group (so the first row per group is the highest-scoring).
    #
    # .head(TOP_K) → DataFrame
    #   Takes the first TOP_K rows from each group (= top-100 per query).
    #
    # .with_columns(rank expression) → DataFrame
    #   Recomputes the rank column after merging:
    #   pl.col("score").rank(method="ordinal", descending=True) → assigns rank 1, 2, 3...
    #     to each row within its group, breaking ties by row order.
    #   .over("query_id") → applies the rank within each query_id partition (like SQL window).
    #   .sub(1) → converts from 1-based to 0-based rank.
    merged_df = (
        all_results
        .sort("score", descending=True)
        .group_by("query_id", maintain_order=True)
        .head(TOP_K)
        .with_columns(
            pl.col("score")
              .rank(method="ordinal", descending=True)
              .over("query_id")
              .sub(1)
              .cast(pl.Int16)
              .alias("rank")
        )
    )
    del all_results

    merged_df.write_parquet(merged_path)
    elapsed = time.time() - t0
    print(f"Saved {merged_path} — {len(merged_df):,} rows in {elapsed:.1f}s")

# summary stats
# .n_unique() → int: count of distinct values in the column
print(f"\nQueries with results: {merged_df['query_id'].n_unique():,}")
print(f"Score range: [{merged_df['score'].min():.4f}, {merged_df['score'].max():.4f}]")
print(f"Shards represented: {sorted(merged_df['shard_id'].unique().to_list())}")
merged_df.head(10)

Loaded shard 00: 3,137,200 rows
Shard 01: not available, skipping
Shard 02: not available, skipping
Shard 03: not available, skipping
Loaded shard 04: 3,137,200 rows
Loaded shard 05: 3,137,200 rows
Loaded shard 06: 3,137,200 rows
Loaded shard 07: 3,137,200 rows
Loaded shard 08: 3,137,200 rows

Merging 6 shards ...
Total candidates: 18,823,200
Saved data\NQ_answer\top100_merged.parquet — 3,137,200 rows in 0.5s

Queries with results: 31,372
Score range: [0.7794, 1.9788]
Shards represented: [0, 4, 5, 6, 7, 8]


query_id,passage_id,score,rank,shard_id
i32,i64,f32,i16,i8
9802,40185194,1.97885,0,8
9802,40185195,1.92371,1,8
9802,40185188,1.498539,2,8
9802,40185193,1.458175,3,8
9802,40185190,1.452949,4,8
9802,40185191,1.442729,5,8
9802,40185189,1.421842,6,8
9802,40185192,1.397847,7,8
9802,41927980,1.388693,8,8


## 8. Validation

Spot-check: pick a sample query, show its top-5 retrieved passages with text from the corpus.

In [11]:
# pick a sample query to inspect
sample_qid = 0
sample_question = queries[sample_qid]["question"]
sample_answers = queries[sample_qid]["answers"]

print(f"Query {sample_qid}: '{sample_question}'")
print(f"Gold answers: {sample_answers}\n")

# get top-5 for this query from merged results
# .filter(expr) → DataFrame: keeps only rows where the boolean expression is True.
# pl.col("query_id") == sample_qid → boolean mask over the column.
top5 = (
    merged_df
    .filter(pl.col("query_id") == sample_qid)
    .sort("rank")
    .head(5)
)

print(f"{'Rank':<6} {'Score':<10} {'Passage ID':<12} {'Shard':<7}")
print("-" * 40)
# .iter_rows(named=True) → Iterator[dict]
#   Yields one dict per row with {column_name: value}.
#   named=False (default) would yield plain tuples instead.
for row in top5.iter_rows(named=True):
    print(f"{row['rank']:<6} {row['score']:<10.4f} {row['passage_id']:<12} {row['shard_id']:<7}")

Query 0: 'where do you cross the arctic circle in norway'
Gold answers: ['Saltfjellet']

Rank   Score      Passage ID   Shard  
----------------------------------------
0      1.1907     34778372     6      
1      1.1654     34778373     6      
2      1.1095     3933046      0      
3      1.0816     1336444      0      
4      1.0613     20343010     4      


In [12]:
# summary: list all output files
print("\nOutput files:")
# Path.rglob(pattern) → Iterator[Path]: recursively matches all files/dirs
#   under this path that match the glob pattern. "*" matches everything.
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        # Path.relative_to(base) → PurePath: strips the base prefix,
        #   returning the relative portion (e.g. "shard_00/top100_shard.parquet").
        print(f"  {f.relative_to(OUTPUT_DIR)!s:<40} {size_mb:>10,.1f} MB")


Output files:
  query_embeddings.npy                           96.4 MB
  shard_00\top100_shard.parquet                  19.6 MB
  shard_04\top100_shard.parquet                  20.0 MB
  shard_05\top100_shard.parquet                  20.0 MB
  shard_06\top100_shard.parquet                  20.4 MB
  shard_07\top100_shard.parquet                  20.0 MB
  shard_08\top100_shard.parquet                  23.9 MB
  top100_merged.parquet                          21.7 MB


---

## 9. Passage Entity Linking with ReFiNed

**Where we are:** After dense retrieval (sections 1–8), before KG reranking (Step 5)  
**What we do:** Extract Wikidata entities (QIDs) from the ~1.5M unique passages in `top100_merged.parquet`  
**How it connects:** The QIDs will be used to compute `kg_score` during reranking

**Why not the full corpus (42M passages)?**  
Only passages that appear in at least one query's top-100 need entity linking.  
This reduces the workload by ~20×.

**Model**: same as `nq_filtering.ipynb` — `questions_model`, entity set `wikipedia`  
**Resume support**: passages are processed in chunks of 10,000; each chunk is saved as a separate parquet file. Re-running skips completed chunks.

### Output

```
data/NQ_answer/
├── passage_entities/
│   ├── chunk_000.parquet      ← (passage_id, title, text, qids)
│   ├── chunk_001.parquet
│   └── ...
└── passage_entities.parquet   ← merged final output
```

In [ ]:
# --- Configuration for passage entity linking ---
CORPUS_PATH   = Path("data/wikipedia_2018_sentence_aligned/psgs_w100_sentence.tsv")
ENTITIES_DIR  = OUTPUT_DIR / "passage_entities"   # chunk storage
ENTITIES_DIR.mkdir(parents=True, exist_ok=True)
ENTITIES_OUT  = OUTPUT_DIR / "passage_entities.parquet"  # final merged output
CHUNK_SIZE    = 10_000  # passages per checkpoint

REFINED_DATA_DIR = Path.cwd() / "data" / "refined_cache"
REFINED_DATA_DIR.mkdir(parents=True, exist_ok=True)

### 9.1 Extract Unique Passage IDs

From `top100_merged.parquet`, collect all distinct passage IDs that need entity linking.

In [ ]:
# reload merged results if not already in memory (e.g. fresh kernel restart)
if "merged_df" not in dir():
    merged_df = pl.read_parquet(OUTPUT_DIR / "top100_merged.parquet")

# .unique() → Series: removes duplicate values, returns distinct passage IDs.
# .sort() → Series: sorts ascending for deterministic chunk ordering across runs.
# .to_numpy() → ndarray: converts to numpy array for fast set-membership lookups later.
unique_pids = merged_df["passage_id"].unique().sort().to_numpy()

print(f"Unique passages to process: {unique_pids.shape[0]:,}")
print(f"Total chunks of {CHUNK_SIZE:,}: {math.ceil(len(unique_pids) / CHUNK_SIZE)}")

### 9.2 Load Passage Texts from Corpus

The corpus TSV has ~42M rows (14.5 GB). We use Polars **lazy scanning** to stream
through the file and keep only the ~1.5M passages we need, without loading everything into RAM.

In [ ]:
print(f"Loading passages from {CORPUS_PATH} ...")
t0 = time.time()

# Convert passage IDs to a Polars Series for use in the filter expression.
# pl.Series(name, values) → Series: wraps a numpy array into a named Polars column.
pid_series = pl.Series("id", unique_pids)

# pl.scan_csv(path, separator, ...) → LazyFrame
#   Creates a LAZY reader: no data is loaded yet.
#   The query plan (filter, select) is optimized before execution.
#   separator="\t": tab-separated file.
#   has_header=True: first row contains column names (id, text, title).
#   schema_overrides: force column types (avoids auto-inference scanning the whole file).
#
# .filter(pl.col("id").is_in(pid_series)) → LazyFrame
#   Adds a filter predicate to the query plan: keep only rows whose "id"
#   is in our set of needed passage IDs. Polars hashes the set for O(1) lookups.
#
# .collect() → DataFrame
#   Executes the lazy query: streams through the TSV, applies the filter on-the-fly,
#   and materializes only the matching rows into RAM.
passages_df = (
    pl.scan_csv(
        CORPUS_PATH,
        separator="\t",
        has_header=True,
        schema_overrides={"id": pl.Int64, "text": pl.String, "title": pl.String},
    )
    .filter(pl.col("id").is_in(pid_series))
    .collect()
)

elapsed = time.time() - t0
print(f"Loaded {len(passages_df):,} passages in {elapsed:.1f}s")
print(f"Coverage: {len(passages_df):,} / {len(unique_pids):,} "
      f"({len(passages_df)/len(unique_pids)*100:.1f}%)")

# sanity: check for missing passages (should be 0 if corpus is complete)
loaded_ids = set(passages_df["id"].to_list())
missing = [pid for pid in unique_pids if pid not in loaded_ids]
if missing:
    print(f"WARNING: {len(missing):,} passage IDs not found in corpus!")
else:
    print("All passage IDs found in corpus.")

passages_df.head(3)

### 9.3 Load ReFiNed Model

Same configuration as `nq_filtering.ipynb`: `questions_model` with `wikipedia` entity set.  
The model uses GPU (transformer-based disambiguation) + CPU (candidate dictionary lookup).

In [ ]:
import subprocess

# Apply source-level patches for ReFiNed V1 compatibility
# (Windows strftime, Python 3.12+ raw strings, transformers 4.x kwargs).
# Safe to re-run: patches are idempotent (skipped if already applied).
subprocess.check_call([sys.executable, "scripts/patch_refined.py"])

from refined.inference.processor import Refined

# Refined.from_pretrained(model_name, entity_set, data_dir) → Refined
#   model_name: which pretrained checkpoint to load.
#     "questions_model" — fine-tuned on WebQSP for question/short-text entity linking.
#     "wikipedia_model" — trained on Wikipedia prose (better for long formal text).
#   entity_set: which entity dictionary to use for candidate generation.
#     "wikipedia" (~6M entities with Wikipedia pages) — sufficient for NQ-open.
#     "wikidata" (~33M entities, ~20 GB) — broader but heavier.
#   data_dir: local cache directory for model weights + entity tables.
#     First call downloads from S3; subsequent calls load from cache.
#   Returns a Refined instance with .process_text() method ready to use.
refined_model = Refined.from_pretrained(
    model_name="questions_model",
    entity_set="wikipedia",
    data_dir=str(REFINED_DATA_DIR),
)
print("ReFiNed model loaded")

### 9.4 Entity Linking Loop (chunked, resumable)

Process passages in chunks of 10,000. Each chunk is saved as a separate parquet file.  
On re-run, completed chunks are skipped automatically.

For each passage we feed `title + " " + text` to ReFiNed — the title provides
disambiguation context (e.g. "Aaron" the biblical figure vs "Aaron" the cricketer).

In [ ]:
def extract_qids(text: str) -> list[str]:
    """Run ReFiNed entity linking on a text string, return Wikidata QIDs.

    refined_model.process_text(text) → list[Span]
      Runs the full entity linking pipeline on the input string:
        1. Mention detection: identifies candidate entity mentions in text
        2. Candidate generation: looks up possible entities per mention (CPU, dictionary)
        3. Entity disambiguation: transformer model scores candidates (GPU, neural)
      Returns a list of Span objects, one per detected mention.

    Each Span has:
      span.text → str: the surface form found in the input (e.g. "Arctic Circle")
      span.predicted_entity → Entity | None: the top-ranked entity, or None if unlinked
        .wikidata_entity_id → str | None: QID (e.g. "Q176609")
        .wikipedia_entity_title → str | None: Wikipedia page title
    """
    spans = refined_model.process_text(text)
    qids = []
    for span in spans:
        ent = span.predicted_entity
        if ent and ent.wikidata_entity_id:
            qids.append(ent.wikidata_entity_id)
    return qids


# --- Sort passages by ID for deterministic chunk assignment ---
# .sort("id") → DataFrame: sorts rows by the "id" column ascending.
# This guarantees the same passage always falls in the same chunk number,
# so resuming after interruption produces identical chunks.
passages_sorted = passages_df.sort("id")

# .to_dicts() → list[dict]: converts each row to a Python dict.
# Needed because we iterate row-by-row for ReFiNed (no batch API).
# This materializes all rows in RAM as dicts — ~1.5M dicts is manageable.
passage_rows = passages_sorted.to_dicts()
del passages_sorted

n_passages = len(passage_rows)
n_chunks = math.ceil(n_passages / CHUNK_SIZE)
print(f"Passages: {n_passages:,} — Chunks: {n_chunks} × {CHUNK_SIZE:,}")

# --- Chunked entity linking with resume ---
t0_total = time.time()
n_skipped = 0
n_processed = 0

for chunk_idx in range(n_chunks):
    chunk_path = ENTITIES_DIR / f"chunk_{chunk_idx:03d}.parquet"

    # resume: skip chunks that already have a saved parquet
    if chunk_path.exists():
        n_skipped += 1
        continue

    start = chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, n_passages)
    chunk_rows = passage_rows[start:end]

    ids = []
    titles = []
    texts = []
    qid_lists = []

    # tqdm(iterable, desc, ...) → iterable wrapper that displays a progress bar.
    #   Updates in real-time with speed (it/s), ETA, and percentage.
    for row in tqdm(chunk_rows, desc=f"Chunk {chunk_idx:03d}/{n_chunks}"):
        # feed title + text together for better disambiguation context
        full_text = f"{row['title']} {row['text']}"
        qids = extract_qids(full_text)

        ids.append(row["id"])
        titles.append(row["title"])
        texts.append(row["text"])
        qid_lists.append(qids)

    # Build and save chunk DataFrame.
    # pl.Series("qids", qid_lists, dtype=pl.List(pl.String)) → Series
    #   Creates a column of type List[String]: each cell is a variable-length list of QIDs.
    #   Polars stores this natively in Arrow list format (no JSON serialization needed).
    chunk_df = pl.DataFrame({
        "id":    pl.Series("id", ids, dtype=pl.Int64),
        "title": pl.Series("title", titles, dtype=pl.String),
        "text":  pl.Series("text", texts, dtype=pl.String),
        "qids":  pl.Series("qids", qid_lists, dtype=pl.List(pl.String)),
    })
    chunk_df.write_parquet(chunk_path)

    n_with_ent = sum(1 for q in qid_lists if q)
    n_processed += len(chunk_rows)
    print(f"  Saved {chunk_path.name} — {len(chunk_rows):,} passages, "
          f"{n_with_ent:,} with entities ({n_with_ent/len(chunk_rows)*100:.1f}%)")

elapsed_total = time.time() - t0_total
print(f"\nDone: {n_processed:,} processed, {n_skipped:,} skipped (cached)")
if n_processed > 0:
    print(f"Time: {elapsed_total:.0f}s ({n_processed/elapsed_total:.1f} passages/s)")

### 9.5 Merge Chunks into Final Parquet

Concatenate all chunk parquets into a single `passage_entities.parquet`.

In [ ]:
if ENTITIES_OUT.exists():
    print(f"Final file already exists: {ENTITIES_OUT}")
    entities_df = pl.read_parquet(ENTITIES_OUT)
else:
    # sorted() + glob("chunk_*.parquet") → list of Paths in alphabetical order.
    # Alphabetical order matches chunk index order (chunk_000, chunk_001, ...).
    chunk_files = sorted(ENTITIES_DIR.glob("chunk_*.parquet"))

    if not chunk_files:
        raise RuntimeError("No chunk files found — run section 9.4 first")

    print(f"Merging {len(chunk_files)} chunk files ...")

    # Read all chunks and concatenate vertically.
    # pl.read_parquet(path) is used per-chunk (not scan) since each chunk is small (~10k rows).
    chunk_dfs = [pl.read_parquet(f) for f in chunk_files]
    entities_df = pl.concat(chunk_dfs)
    del chunk_dfs

    entities_df.write_parquet(ENTITIES_OUT)
    print(f"Saved {ENTITIES_OUT} — {len(entities_df):,} passages")

# --- Summary statistics ---
# .list.len() → Series[UInt32]: for a List column, returns the length of each list.
#   e.g. if qids = ["Q30", "Q42"] → list.len() = 2
n_with_entities = entities_df.filter(
    pl.col("qids").list.len() > 0
).shape[0]

# .explode("qids") → DataFrame: unnests the list column, creating one row per QID.
#   A passage with qids=["Q30","Q42"] becomes 2 rows. Passages with qids=[] are dropped.
# .n_unique() counts distinct QIDs across all passages.
n_unique_qids = entities_df.explode("qids").drop_nulls("qids")["qids"].n_unique()

print(f"\nTotal passages:      {len(entities_df):,}")
print(f"With ≥1 entity:      {n_with_entities:,} ({n_with_entities/len(entities_df)*100:.1f}%)")
print(f"Without entities:    {len(entities_df) - n_with_entities:,}")
print(f"Unique QIDs found:   {n_unique_qids:,}")

# distribution of entities per passage
qid_counts = entities_df.select(pl.col("qids").list.len().alias("n_qids"))
print(f"\nEntities per passage:")
# .describe() → DataFrame: summary statistics (mean, std, min, max, quantiles).
print(qid_counts.describe())

entities_df.head(5)

### 9.6 Spot-Check

Verify entity linking on the same sample query from section 8: show its top-5 passages with extracted entities.

In [ ]:
# reuse sample_qid=0 from section 8
sample_qid = 0
sample_question = queries[sample_qid]["question"]
sample_answers = queries[sample_qid]["answers"]
sample_q_qids = queries[sample_qid].get("question_qids", [])

print(f"Query {sample_qid}: '{sample_question}'")
print(f"Gold answers: {sample_answers}")
print(f"Question QIDs: {sample_q_qids}\n")

# get top-5 passage IDs for this query
top5_pids = (
    merged_df
    .filter(pl.col("query_id") == sample_qid)
    .sort("rank")
    .head(5)
)

# join with entity data to show QIDs alongside passages
# .join(other, on, how) → DataFrame
#   Combines two DataFrames on a shared key column.
#   on="passage_id"/"id": the join key (renamed to match).
#   how="left": keep all rows from the left DF; fill with null if no match on right.
top5_with_ent = top5_pids.join(
    entities_df.rename({"id": "passage_id"}),
    on="passage_id",
    how="left",
)

for row in top5_with_ent.iter_rows(named=True):
    print(f"Rank {row['rank']} | Score {row['score']:.4f} | PID {row['passage_id']}")
    print(f"  Title: {row['title']}")
    # truncate long text for display
    text_preview = row["text"][:150] + "..." if len(row["text"]) > 150 else row["text"]
    print(f"  Text:  {text_preview}")
    print(f"  QIDs:  {row['qids']}")
    print()